# PCA-based Image Compression for Simulation Data

In this notebook, we apply Principal Component Analysis (PCA) to compress six density maps (three period-period derivative maps plus three flux maps in the same space) for a given synthetic simulation into a single latent vector. In the following, we fit the PCA model on a dataset containing $N=20$ synthetic pulsar simulations and select enough components to capture 95% of the variance in the data. The resulting compressed vectors could, e.g., be used as summary statistics in Neural Likelihood Estimation (NLE) as an alternative to CNNs to estimate the likelihood distribution given the observed data. 

For a general tutorial on PCA, we refer to https://jakevdp.github.io/PythonDataScienceHandbook/05.09-principal-component-analysis.html.

In [ ]:
import json
import os
import torch
import joblib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mlpoppyns.learning.loaders.loader_multichannel_array as dl
from mlpoppyns.generator import generate_dataset_surveys
from sklearn.decomposition import PCA
from mlpoppyns.learning.utils import sbi_utils
from typing import Tuple

import logging

logger = logging.getLogger("logger")
logger.setLevel(logging.INFO)

# Add a console handler (optional).
handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

We first specify the path to the dataset used for extracting the PCA components, as well as the path to the observed sample. This allows evaluation of how well the PCA-compressed representation works for the observed data.
The `config_sbi.json` file must also be provided to correctly select the relevant maps from `dataset_full.csv`.

In [ ]:
training_dataset_path = "../../data/example_generator_magrot"
observed_dataset_path = "../../data/example_generator_observed"
config_path = "../../mlpoppyns/learning/config_sbi.json"

In [ ]:
with open(config_path, "r") as file:
    config = json.load(file)

We need to set use_compression = False and set the model to snpe, because we want to generate a dataset of density images. Compression must be disabled, and snpe is the only model that supports density map representation. Even though the model type is not explicitly used here, specifying a different one would raise an error.

In [ ]:
config["compression_input"]["use_compression"] = False
config["trainer"]["type"] = 'snpe'

In [ ]:
dataset, param, matrix = sbi_utils.prepare_dataset_sbi(
    training_dataset_path, config, logger, atnf=False
)
_, _, matrix_obs = sbi_utils.prepare_dataset_sbi(
    observed_dataset_path, config, logger, atnf=True
)

In [ ]:
matrix.shape

Flatten the six density maps with a resolution of 32x32 each into a single vector of length 6144 (for each of the 20 synthetic simulations) to apply PCA. 

In [ ]:
matrix_reshaped = matrix.reshape(matrix.shape[0], -1)
matrix_reshaped.shape

Calculate and plot the cumulative explained variance for different numbers of principal components.

In [ ]:
pca = PCA().fit(matrix_reshaped)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Full plot (all components)
axs[0].plot(cumulative_variance)
axs[0].set_title("Full View (All Components)")
axs[0].set_xlabel("Number of Components")
axs[0].set_ylabel("Cumulative Explained Variance")
axs[0].grid()

# Zoomed-in plot (first 10 components)
axs[1].plot(cumulative_variance[:10])
axs[1].set_title("Zoomed-in View (First 10 Components)")
axs[1].set_xlabel("Number of Components")
axs[1].set_ylabel("Cumulative Explained Variance")
axs[1].set_xticks(np.arange(0,10,1))
axs[1].grid()


plt.tight_layout()
plt.show()

The plot above shows the cumulative variance explained as a function of the number of components. As illustrated in the zoomed-in plot on the right, 10 components are sufficient to retain 95% of the total variance. Therefore, to preserve this level of variance, the compressed representation is reduced to 10 dimensions. We then save the fitted PCA model and compute the compressed vectors for both a randomly selected simulation and the observed data.

In [ ]:
variance = 0.95
pca = PCA(n_components=variance)
pca.fit(matrix_reshaped)

In [ ]:
pca_saved_path = "1_pca_model_95_variance.pkl"
joblib.dump(pca, pca_saved_path)

In [ ]:
def embedding_pca(
    matrix: np.ndarray, path_trained_pca: str, plot: bool = True
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compress and reconstruct a set of six density maps using a pre-trained PCA model.

    This function applies a Principal Component Analysis (PCA) transformation to a
    combined vector representation of six 2D density maps. It compresses the input
    into a lower-dimensional latent vector and reconstructs the original maps
    from that compressed representation. Optionally, it visualizes the original
    and reconstructed maps for comparison.

    Arguments:
        matrix (np.ndarray): Array of shape (6, 1024), representing six flattened
            32x32 density maps.
        path_trained_pca (str): Path to the directory containing the saved PCA model.
        plot (bool): Whether to display a side-by-side comparison plot of the
            original and reconstructed maps.

    Returns:
        compressed_vector (np.ndarray): The compressed representation of the input
            maps (i.e., PCA-transformed vector).
        reconstructed_maps (np.ndarray): Array of shape (6, 32, 32) representing
            the reconstructed density maps.
    """
    pca = joblib.load(f"{path_trained_pca}")
    flattened = matrix.reshape(1, -1)  # shape (1, 6144)

    compressed = pca.transform(flattened)
    reconstructed = pca.inverse_transform(compressed).reshape(6, 32, 32)

    if plot:
        fig, ax = plt.subplots(
            2, 6, figsize=(12, 4), subplot_kw={"xticks": [], "yticks": []}
        )
        for i in range(6):
            ax[0, i].imshow(matrix[i].reshape(32, 32), cmap="binary_r")
            ax[1, i].imshow(reconstructed[i], cmap="binary_r")
        ax[0, 0].set_ylabel("Original")
        ax[1, 0].set_ylabel("Reconstructed")
        plt.suptitle("Single-PCA Compression & Reconstruction")
        plt.show()

    return compressed, reconstructed

Select a random sample to compare the compressed representation after reconstruction with the original images.

In [ ]:
random_index = np.random.randint(0, matrix.shape[0])

In [ ]:
new_sample = matrix[random_index, :, :]

matrix_compressed, reconstructed = embedding_pca(
    new_sample, pca_saved_path, plot=True
)

In [ ]:
matrix_compressed, reconstructed = embedding_pca(
   new_sample, pca_saved_path, plot=True
)

Compare the compressed representation after the reconstruction with the original for the observed data.

In [ ]:
matrix_compressed, reconstructed = embedding_pca(
    matrix_obs[0], pca_saved_path, plot=True
)

In the two plots above, we show the original images for a random test sample and for the observed sample, alongside their corresponding reconstructions from the PCA vectors. That is, we take the compressed vector produced by PCA and apply the inverse transformation to reconstruct the image. If we retained 100% of the variance, the original and reconstructed images would be identical. However, since we only preserve 95% of the variance, the reconstructed images appear slightly blurrier.